In [7]:
!pip install ultralytics

In [8]:
!nvidia-smi

Sat Feb 14 08:48:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             25W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
SOURCE = "/kaggle/input/datasets/moltean/fruits/fruits-360_100x100/fruits-360/Training"
DEST = "/kaggle/working/Not_Pomegranate"

import os, shutil

os.makedirs(DEST, exist_ok=True)

for folder in os.listdir(SOURCE):

    # skip pomegranate folders
    if "pomegranate" in folder.lower():
        continue

    path = os.path.join(SOURCE, folder)

    for img in os.listdir(path)[:300]:  # limit per fruit
        shutil.copy(
            os.path.join(path, img),
            os.path.join(DEST, img)
        )

print("Not_Pomegranate dataset created successfully")


Not_Pomegranate dataset created successfully


In [10]:
len(os.listdir("/kaggle/working/Not_Pomegranate"))


1600

In [11]:
import shutil
import os

SOURCE_MAIN = "/kaggle/input/pomegranate-fruit-diseases-dataset/Pomegranate Fruit Diseases Dataset for Deep Learning Models/Pomegranate Diseases Dataset/Pomegranate Diseases Dataset"
WORKING_MAIN = "/kaggle/working/dataset"

# remove existing folder if it exists
if os.path.exists(WORKING_MAIN):
    shutil.rmtree(WORKING_MAIN)

# copy fresh
shutil.copytree(SOURCE_MAIN, WORKING_MAIN)

print("Main dataset copied successfully")


Main dataset copied successfully


In [12]:
shutil.copytree(
    "/kaggle/working/Not_Pomegranate",
    f"{WORKING_MAIN}/Not_Pomegranate",
    dirs_exist_ok=True
)


'/kaggle/working/dataset/Not_Pomegranate'

In [13]:
import os
print(os.listdir(WORKING_MAIN))


['Alternaria', 'Not_Pomegranate', 'Healthy', 'Anthracnose', 'Cercospora', 'Bacterial_Blight']


In [14]:
import os, random, shutil

SOURCE = "/kaggle/working/dataset"
DEST = "/kaggle/working/yolo_dataset"

classes = sorted(os.listdir(SOURCE))
class_map = {c:i for i,c in enumerate(classes)}

print("Classes:", class_map)

for split in ["train","val"]:
    os.makedirs(f"{DEST}/images/{split}", exist_ok=True)
    os.makedirs(f"{DEST}/labels/{split}", exist_ok=True)

for cls in classes:
    files = os.listdir(f"{SOURCE}/{cls}")
    random.shuffle(files)

    split_idx = int(len(files)*0.8)

    for i,f in enumerate(files):
        split = "train" if i<split_idx else "val"

        shutil.copy(
            f"{SOURCE}/{cls}/{f}",
            f"{DEST}/images/{split}/{f}"
        )

        with open(f"{DEST}/labels/{split}/{f.replace('.jpg','.txt')}","w") as w:
            w.write(f"{class_map[cls]} 0.5 0.5 1 1\n")

print("YOLO dataset created")


Classes: {'Alternaria': 0, 'Anthracnose': 1, 'Bacterial_Blight': 2, 'Cercospora': 3, 'Healthy': 4, 'Not_Pomegranate': 5}
YOLO dataset created


In [15]:
yaml = f"""
path: {DEST}
train: images/train
val: images/val

names:
"""

for i,c in enumerate(classes):
    yaml += f"  {i}: {c}\n"

open("/kaggle/working/data.yaml","w").write(yaml)

print(yaml)



path: /kaggle/working/yolo_dataset
train: images/train
val: images/val

names:
  0: Alternaria
  1: Anthracnose
  2: Bacterial_Blight
  3: Cercospora
  4: Healthy
  5: Not_Pomegranate



In [16]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=40,
    imgsz=640,
    batch=16
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=

KeyboardInterrupt: 

In [ ]:
import shutil

shutil.copy(
    "runs/detect/train/weights/best.pt",
    "/kaggle/working/best.pt"
)

print("best.pt saved")


In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")
metrics = model.val()
print(metrics)


In [17]:
from ultralytics import YOLO

model = YOLO("/kaggle/input/models/vikrammugale/yolov11-pomegranate-disease-classifier/pytorch/default/1/best.pt")


print("Classes:", model.names)


Classes: {0: 'Alternaria', 1: 'Anthracnose', 2: 'Bacterial_Blight', 3: 'Cercospora', 4: 'Healthy', 5: 'Not_Pomegranate'}


In [18]:
results = model.predict("/kaggle/input/pomegranate-fruit-diseases-dataset/Pomegranate Fruit Diseases Dataset for Deep Learning Models/Pomegranate Diseases Dataset/Pomegranate Diseases Dataset/Anthracnose/IMG_20230813_151602.jpg", conf=0.25)

r = results[0]

if len(r.boxes):
    cls = int(r.boxes.cls[0])
    conf = float(r.boxes.conf[0])
    print("Prediction:", model.names[cls], conf)
else:
    print("No detection")



image 1/1 /kaggle/input/pomegranate-fruit-diseases-dataset/Pomegranate Fruit Diseases Dataset for Deep Learning Models/Pomegranate Diseases Dataset/Pomegranate Diseases Dataset/Anthracnose/IMG_20230813_151602.jpg: 640x640 1 Anthracnose, 8.2ms
Speed: 3.6ms preprocess, 8.2ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)
Prediction: Anthracnose 0.9919773936271667


In [19]:
import os

DATASET = "/kaggle/working/dataset"  # change if needed

for cls in sorted(os.listdir(DATASET)):
    path = os.path.join(DATASET, cls)
    if os.path.isdir(path):
        print(f"{cls:20} -> {len(os.listdir(path))}")


Alternaria           -> 886
Anthracnose          -> 1166
Bacterial_Blight     -> 966
Cercospora           -> 631
Healthy              -> 1450
Not_Pomegranate      -> 1600


In [20]:
import os
import random
import shutil

DATASET = "/kaggle/working/dataset"

TARGET = {
    "Alternaria": 1000,
    "Anthracnose": 1000,
    "Bacterial_Blight": 1000,
    "Cercospora": 1000,
    "Healthy": 1000,
    "Not_Pomegranate": 1200
}

for cls, target in TARGET.items():
    cls_path = os.path.join(DATASET, cls)
    images = os.listdir(cls_path)

    print(f"\nBalancing {cls}")

    # oversample if too small
    while len(images) < target:
        img = random.choice(images)
        src = os.path.join(cls_path, img)
        new_name = f"aug_{len(images)}_{img}"
        dst = os.path.join(cls_path, new_name)
        shutil.copy(src, dst)
        images.append(new_name)

    # undersample if too big
    if len(images) > target:
        remove = random.sample(images, len(images) - target)
        for img in remove:
            os.remove(os.path.join(cls_path, img))

print("\nRebalancing complete.")



Balancing Alternaria

Balancing Anthracnose

Balancing Bacterial_Blight

Balancing Cercospora

Balancing Healthy

Balancing Not_Pomegranate

Rebalancing complete.


In [22]:
from ultralytics import YOLO

model = YOLO("/kaggle/input/models/vikrammugale/yolov11-pomegranate-disease-classifier/pytorch/default/1/best.pt")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=40,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    patience=20
)


Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/models/vikrammugale/yolov11-pomegranate-disease-classifier/pytorch/default/1/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a65756d3860>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [23]:
import shutil

shutil.copy(
    "/kaggle/working/runs/detect/train2/weights/best.pt",
    "/kaggle/working/final_model.pt"
)


'/kaggle/working/final_model.pt'